In [47]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [48]:
from helper import DataLoader
import models.SimpleGCN.DualGraphGCN as DualGraphGCN
import models.SimpleGCN.SimpleGCN as SimpleGCN
import torch
from sklearn.metrics import root_mean_squared_error

In [49]:
DATAPATH = "../data"
RATING_DATA = DATAPATH + "/train_ratings.csv"
TBR_DATA = DATAPATH + "/train_tbr.csv"

In [50]:
data_manager = DataLoader.DataLoader(data_dir=DATAPATH)
train_df, valid_df = data_manager.load_and_split()

In [51]:
print(train_df)

         sid  pid  rating
0          0    1       5
1          0   13       4
2          0   22       5
3          0   26       4
4          0   62       4
...      ...  ...     ...
896738   957  337       3
896739  2218  232       5
896740  1146  255       4
896741  5758  519       4
896742  1062  437       5

[896743 rows x 3 columns]


In [52]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [53]:
edge_t0_index, edge_t0_weights, edge_t1_index, edge_t1_weights = data_manager.get_graph_tensors(device)

In [54]:
# model = DualGraphGCN.DualGraphGCN(
#     num_users=data_manager.num_users,
#     num_items=data_manager.num_items,
#     embedding_dim=64,
#     num_layers=2,
#     dropout=0.5
# ).to(device)

# # model = SimpleGCN.SimpleGCN(
# #     num_users=data_manager.num_users,
# #     num_items=data_manager.num_items,
# #     embedding_dim=64,
# #     num_layers=1,
# #     dropout=0.3
# # ).to(device)

# criterion = torch.nn.MSELoss()
# def predict_ratings(sids, pids):
#     model.eval()
#     with torch.no_grad():
#         s_tensor = torch.tensor(sids, dtype=torch.long).to(device)
#         p_tensor = torch.tensor(pids, dtype=torch.long).to(device)
#         # Pass all 4 graph tensors
#         out, _, _ = model(s_tensor, p_tensor, 
#                    edge_t0_index, edge_t0_weights, 
#                    edge_t1_index, edge_t1_weights)        
#         # out, _, _ = model(s_tensor, p_tensor, 
#         #             edge_t0_index, edge_t0_weights)
#         return out.cpu().numpy()

In [55]:
# optimizer = torch.optim.Adam(
#     model.parameters(),
#     lr=1e-2,
# )

In [ ]:
# EPOCHS = 5000
# LAMBDA_REG = 1e-4
# LAMBDA_WISHLIST = 0.5

# # --- 3. UPDATED TRAINING LOOP ---
# for epoch in range(EPOCHS):
#     model.train()
#     optimizer.zero_grad()
    
#     # Forward pass using both relation types
#     # We predict ratings for the users/items present in our t0 (ratings) edges
    
#     preds, (u_w, i_w), (u_init_r, i_init_r, u_init_w, i_init_w) = model(
#         edge_t0_index[0], edge_t0_index[1], 
#         edge_t0_index, edge_t0_weights,
#         edge_t1_index, edge_t1_weights
#     )    
#     pred_wishlist = (u_w[edge_t1_index[0]] * i_w[edge_t1_index[1]]).sum(-1)
#     # preds, u_init, i_init = model(
#     #     edge_t0_index[0],edge_t0_index[1], edge_t0_index, edge_t0_weights
#     # )
    
#     # Loss is still calculated against the actual ratings (weights_t0)
#     mse_loss = criterion(preds, edge_t0_weights * 5)
#     mse_loss_wishlist = criterion(pred_wishlist, edge_t1_weights * 1)
#     # l2_reg = (u_init.norm(2)**2 + i_init.norm(2)**2) / len(edge_t1_index[0])

#     loss = (
#         mse_loss 
#         + LAMBDA_WISHLIST * mse_loss_wishlist   # auxiliary
#         + LAMBDA_REG * sum(e.norm() for e in [u_init_r, i_init_r, u_init_w, i_init_w])  # regularization
#     )

    
#     loss.backward()
#     optimizer.step()
    
#     if epoch % 1 == 0:
#         val_preds = predict_ratings(valid_df["sid"].values, valid_df["pid"].values)
#         val_rmse = root_mean_squared_error(valid_df["rating"].values, val_preds)
#         print(f"Epoch {epoch} | Loss: {loss.item():.4f} | MSE: {mse_loss.item():.4f} | MSE Wishlist: {mse_loss_wishlist.item():.4f} | Val RMSE: {val_rmse:.4f}")


In [66]:
rating_model = SimpleGCN.SimpleGCN(data_manager.num_users, data_manager.num_items, embedding_dim=32, num_layers=1, layer_type="LightGCN")

def rating_val():
    rating_model.eval()
    with torch.no_grad():
        # 1. Pass the IDs AND the graph tensors
        val_preds = rating_model.predict_ratings(
            valid_df["sid"].values, 
            valid_df["pid"].values,
            edge_t0_index,    # The graph structure
            edge_t0_weights   # The graph weights
        )
        
        # 2. Ensure variable names match (val_preds)
        return root_mean_squared_error(valid_df["rating"].values, val_preds * 5)


rating_model.fit(
    edge_index=edge_t0_index,
    weights=edge_t0_weights,
    log_every=10,
    lr=1e-2,
    val_fn=rating_val,
    targets=edge_t0_weights.float(),
    lambda_reg=1e-4
)

Epoch     0 | Loss: 0.6260 | Task: 0.6260 | Monitor: 3.9452 | LR: 1.00e-02 | No improvement: 0/100
Epoch    10 | Loss: 0.2152 | Task: 0.2152 | Monitor: 1.9636 | LR: 1.00e-02 | No improvement: 0/100
Epoch    20 | Loss: 0.1254 | Task: 0.1254 | Monitor: 1.5763 | LR: 1.00e-02 | No improvement: 7/100
Epoch    30 | Loss: 0.0786 | Task: 0.0786 | Monitor: 1.3755 | LR: 1.00e-02 | No improvement: 7/100
Epoch    40 | Loss: 0.0519 | Task: 0.0519 | Monitor: 1.1512 | LR: 1.00e-02 | No improvement: 3/100
Epoch    50 | Loss: 0.0417 | Task: 0.0417 | Monitor: 1.0212 | LR: 1.00e-02 | No improvement: 2/100
Epoch    60 | Loss: 0.0381 | Task: 0.0381 | Monitor: 0.9645 | LR: 1.00e-02 | No improvement: 0/100
Epoch    70 | Loss: 0.0367 | Task: 0.0367 | Monitor: 0.9440 | LR: 1.00e-02 | No improvement: 0/100
Epoch    80 | Loss: 0.0359 | Task: 0.0359 | Monitor: 0.9367 | LR: 1.00e-02 | No improvement: 0/100
Epoch    90 | Loss: 0.0358 | Task: 0.0358 | Monitor: 0.9348 | LR: 1.00e-02 | No improvement: 1/100
Epoch   10

KeyboardInterrupt: 